# Sunburn Cases vs UV Index: A Statistical Assessment

This notebook provides a comprehensive analysis of the relationship between UV Index and emergency department sunburn/heat-stroke cases in England. We analyze UK Health Security Agency (UKHSA) data to determine the core relationship between UV exposure and sunburn cases.

## Key Objectives
1. Explore the temporal patterns of UV index and sunburn cases
2. Visualize the relationship through multiple detailed graphs
3. Perform hypothesis testing to statistically validate the UV-sunburn relationship
4. Accept or reject hypotheses about \relationships

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, f_oneway, kruskal
import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

project_root = Path.cwd()
print(f"Project root: {project_root}")
print("Libraries loaded successfully!")

In [ ]:
weekly_data_path = project_root / 'tmp' / 'uk_sunburns' / 'data' / 'weekly_aggregated_data.csv'
df = pd.read_csv(weekly_data_path)

df['week_start'] = pd.to_datetime(df['year_week'].str.split('/').str[0])
df['month'] = df['week_start'].dt.month
df['month_name'] = df['week_start'].dt.strftime('%B')
df['year'] = df['week_start'].dt.year
df['season'] = df['month'].apply(lambda x: 'Winter' if x in [12, 1, 2] else 
                                           'Spring' if x in [3, 4, 5] else 
                                           'Summer' if x in [6, 7, 8] else 'Autumn')

print(f"Data loaded: {len(df)} weeks")
print(f"Date range: {df['week_start'].min().strftime('%Y-%m-%d')} to {df['week_start'].max().strftime('%Y-%m-%d')}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print("="*70)
print("SUMMARY STATISTICS")
print("="*70)

print("\nUV Index Statistics (Weekly Average):")
print(f"   Mean: {df['avg_uv_index'].mean():.3f}")
print(f"   Std:  {df['avg_uv_index'].std():.3f}")
print(f"   Min:  {df['avg_uv_index'].min():.3f}")
print(f"   Max:  {df['avg_uv_index'].max():.3f}")
print(f"   Median: {df['avg_uv_index'].median():.3f}")

print("\nSunburn Cases Statistics (Weekly Total):")
print(f"   Mean: {df['total_cases'].mean():.2f}")
print(f"   Std:  {df['total_cases'].std():.2f}")
print(f"   Min:  {df['total_cases'].min():.0f}")
print(f"   Max:  {df['total_cases'].max():.0f}")
print(f"   Median: {df['total_cases'].median():.0f}")
print(f"   Total: {df['total_cases'].sum():.0f}")

print("\nSeasonal Distribution of Cases:")
seasonal_stats = df.groupby('season')['total_cases'].agg(['sum', 'mean', 'count'])
print(seasonal_stats.to_string())

## 3. Detailed Visualizations

### 3.1 Time Series: UV Index vs Sunburn Cases

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 7))

color_uv = '#FF6B35'
ax1.fill_between(df['week_start'], df['avg_uv_index'], alpha=0.3, color=color_uv)
ax1.plot(df['week_start'], df['avg_uv_index'], color=color_uv, linewidth=2, label='Avg UV Index')
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Average UV Index', color=color_uv, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color_uv)
ax1.set_ylim(0, df['max_uv_index'].max() * 1.1)

ax2 = ax1.twinx()
color_cases = '#004E89'
ax2.bar(df['week_start'], df['total_cases'], alpha=0.6, color=color_cases, width=5, label='Sunburn Cases')
ax2.set_ylabel('Weekly Sunburn/Heat-stroke Cases', color=color_cases, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color_cases)

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)

plt.title('Weekly UV Index & Sunburn Cases in England', fontsize=16, fontweight='bold', pad=20)
fig.tight_layout()

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.savefig(project_root / 'output' / 'sunburn_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Correlation Scatter Plot with Regression

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

slope, intercept, r_value, p_value, std_err = stats.linregress(df['avg_uv_index'], df['total_cases'])
x_line = np.linspace(df['avg_uv_index'].min(), df['avg_uv_index'].max(), 100)
y_line = slope * x_line + intercept

season_colors = {'Winter': '#1E88E5', 'Spring': '#43A047', 'Summer': '#FFC107', 'Autumn': '#E53935'}
for season, color in season_colors.items():
    mask = df['season'] == season
    ax.scatter(df.loc[mask, 'avg_uv_index'], df.loc[mask, 'total_cases'], 
               c=color, s=100, alpha=0.7, label=season, edgecolors='white', linewidth=0.5)

ax.plot(x_line, y_line, 'k--', linewidth=2, label=f'Regression (r={r_value:.3f})')

from scipy import stats as sp_stats
n = len(df)
y_err = std_err * np.sqrt(1/n + (x_line - df['avg_uv_index'].mean())**2 / ((n-1)*df['avg_uv_index'].var()))
t_val = sp_stats.t.ppf(0.975, n-2)
ax.fill_between(x_line, y_line - t_val*y_err*20, y_line + t_val*y_err*20, alpha=0.2, color='gray')

ax.set_xlabel('Average Weekly UV Index', fontsize=12)
ax.set_ylabel('Weekly Sunburn/Heat-stroke Cases', fontsize=12)
ax.set_title(f'Correlation: UV Index vs Sunburn Cases (r = {r_value:.3f}, p = {p_value:.2e})', 
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(project_root / 'output' / 'sunburn_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n Pearson correlation: r = {r_value:.4f}, p-value = {p_value:.2e}")
print(f"   R² = {r_value**2:.4f} ({r_value**2*100:.1f}% of variance explained)")

### 3.3 Monthly Heatmap of Sunburn Cases

In [ ]:
df['week_of_year'] = df['week_start'].dt.isocalendar().week

monthly_data = df.groupby(['year', 'month']).agg({
    'total_cases': 'sum',
    'avg_uv_index': 'mean'
}).reset_index()

pivot_cases = monthly_data.pivot(index='year', columns='month', values='total_cases')
pivot_uv = monthly_data.pivot(index='year', columns='month', values='avg_uv_index')

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.heatmap(pivot_cases, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0], 
            cbar_kws={'label': 'Total Cases'})
axes[0].set_title('Monthly Sunburn Cases by Year', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Month', fontsize=12)
axes[0].set_ylabel('Year', fontsize=12)
axes[0].set_xticklabels([month_names[int(i)-1] for i in pivot_cases.columns], rotation=45)

sns.heatmap(pivot_uv, annot=True, fmt='.2f', cmap='YlOrBr', ax=axes[1],
            cbar_kws={'label': 'Avg UV Index'})
axes[1].set_title('Monthly Average UV Index by Year', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month', fontsize=12)
axes[1].set_ylabel('Year', fontsize=12)
axes[1].set_xticklabels([month_names[int(i)-1] for i in pivot_uv.columns], rotation=45)

plt.tight_layout()
plt.savefig(project_root / 'output' / 'sunburn_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Hypothesis Testing

We now formally test four hypotheses about the UV-sunburn relationship.

### Hypothesis 1: UV Index and Sunburn Cases are Correlated

**H₀ (Null)**: There is no correlation between UV index and sunburn cases (ρ = 0)  
**H₁ (Alternative)**: There is a significant positive correlation (ρ > 0)

In [ ]:
print("="*70)
print("HYPOTHESIS TEST 1: UV-Sunburn Correlation")
print("="*70)

pearson_r, pearson_p = pearsonr(df['avg_uv_index'], df['total_cases'])

spearman_r, spearman_p = spearmanr(df['avg_uv_index'], df['total_cases'])

alpha = 0.05

print("\n Pearson Correlation Test:")
print(f"   Correlation coefficient (r): {pearson_r:.4f}")
print(f"   P-value: {pearson_p:.2e}")
print(f"   R² (variance explained): {pearson_r**2:.4f} ({pearson_r**2*100:.1f}%)")

print("\n Spearman Correlation Test (Non-parametric):")
print(f"   Correlation coefficient (ρ): {spearman_r:.4f}")
print(f"   P-value: {spearman_p:.2e}")

print("\n" + "-"*70)
print(f"Significance level (α): {alpha}")

if pearson_p < alpha and spearman_p < alpha:
    print("\n RESULT: REJECT H₀")
    print("   Both tests indicate a statistically significant positive correlation.")
    print(f"   The UV index explains approximately {pearson_r**2*100:.1f}% of the variance")
    print("   in sunburn cases.")
    h1_result = "REJECTED"
else:
    print("\n RESULT: FAIL TO REJECT H₀")
    print("   Insufficient evidence to conclude a significant correlation.")
    h1_result = "NOT REJECTED"

### Hypothesis 2: Summer Has Higher Sunburn Rates

**H₀ (Null)**: Sunburn rates are equal across all seasons  
**H₁ (Alternative)**: At least one season has significantly different sunburn rates

In [ ]:
print("="*70)
print("HYPOTHESIS TEST 2: Seasonal Differences in Sunburn Rates")
print("="*70)

winter = df[df['season'] == 'Winter']['total_cases'].values
spring = df[df['season'] == 'Spring']['total_cases'].values
summer = df[df['season'] == 'Summer']['total_cases'].values
autumn = df[df['season'] == 'Autumn']['total_cases'].values

f_stat, anova_p = f_oneway(winter, spring, summer, autumn)

kruskal_stat, kruskal_p = kruskal(winter, spring, summer, autumn)

print("\n One-Way ANOVA Test:")
print(f"   F-statistic: {f_stat:.4f}")
print(f"   P-value: {anova_p:.2e}")

print("\n Kruskal-Wallis Test (Non-parametric):")
print(f"   H-statistic: {kruskal_stat:.4f}")
print(f"   P-value: {kruskal_p:.2e}")

print("\n Group Means:")
print(f"   Winter: {winter.mean():.2f} cases/week (n={len(winter)})")
print(f"   Spring: {spring.mean():.2f} cases/week (n={len(spring)})")
print(f"   Summer: {summer.mean():.2f} cases/week (n={len(summer)})")
print(f"   Autumn: {autumn.mean():.2f} cases/week (n={len(autumn)})")

print("\n" + "-"*70)
print(f"Significance level (α): {alpha}")

if anova_p < alpha:
    print("\n RESULT: REJECT H₀")
    print("   There are statistically significant differences in sunburn rates across seasons.")
    print(f"   Summer has the highest average ({summer.mean():.1f} cases/week).")
    h2_result = "REJECTED"
else:
    print("\n RESULT: FAIL TO REJECT H₀")
    print("   No significant seasonal differences detected.")
    h2_result = "NOT REJECTED"

### Hypothesis 3: UV Index Can Predict Sunburn Cases (Regression)

**H₀ (Null)**: UV index has no predictive power for sunburn cases (β = 0)  
**H₁ (Alternative)**: UV index significantly predicts sunburn cases (β ≠ 0)

In [ ]:
print("="*70)
print("HYPOTHESIS TEST 4: Linear Regression Analysis")
print("="*70)

X = df['avg_uv_index'].values
y = df['total_cases'].values

X_const = sm.add_constant(X)

model = sm.OLS(y, X_const).fit()

print("\n OLS Regression Results:")
print(f"   Dependent Variable: Weekly Sunburn Cases")
print(f"   Independent Variable: Average UV Index")
print(f"   Number of observations: {len(df)}")

print("\n   Model Coefficients:")
print(f"     Intercept (β₀): {model.params[0]:.4f}")
print(f"     UV Index slope (β₁): {model.params[1]:.4f}")
print(f"     β₁ Std Error: {model.bse[1]:.4f}")
print(f"     β₁ t-statistic: {model.tvalues[1]:.4f}")
print(f"     β₁ p-value: {model.pvalues[1]:.2e}")

print("\n   Model Fit Statistics:")
print(f"     R²: {model.rsquared:.4f}")
print(f"     Adjusted R²: {model.rsquared_adj:.4f}")
print(f"     F-statistic: {model.fvalue:.4f}")
print(f"     F-test p-value: {model.f_pvalue:.2e}")

print("\n" + "-"*70)
print(f"Significance level (α): {alpha}")

if model.pvalues[1] < alpha:
    print("\n RESULT: REJECT H₀")
    print(f"   UV Index is a significant predictor of sunburn cases.")
    print(f"   For each 1-unit increase in UV index, sunburn cases increase by ~{model.params[1]:.1f}.")
    print(f"   The model explains {model.rsquared*100:.1f}% of variance in sunburn cases.")
    h4_result = "REJECTED"
else:
    print("\n RESULT: FAIL TO REJECT H₀")
    print("   UV Index is not a significant predictor.")
    h4_result = "NOT REJECTED"

print("\n Residual Diagnostics:")
residuals = model.resid
print(f"   Mean of residuals: {residuals.mean():.4f} (should be ~0)")
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"   Shapiro-Wilk test for normality: p = {shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("    Residuals appear normally distributed")
else:
    print("   ⚠ Residuals may not be normally distributed")

## 5. Summary of Results

In [ ]:
print("="*70)
print("HYPOTHESIS TESTING SUMMARY")
print("="*70)

results_table = f"""
┌─────┬────────────────────────────────────────────┬───────────────┬──────────────┐
│ H
├─────┼────────────────────────────────────────────┼───────────────┼──────────────┤
│ H1  │ UV-Sunburn Correlation                     │ {h1_result:<13} │ {pearson_p:.2e}   │
│ H2  │ Seasonal Differences                       │ {h2_result:<13} │ {anova_p:.2e}   │
│ H3  │ UV Predicts Sunburn (Regression)           │ {h4_result:<13} │ {model.pvalues[1]:.2e}   │
└─────┴────────────────────────────────────────────┴───────────────┴──────────────┘
"""
print(results_table)

print("\n KEY FINDINGS:")
print("─" * 70)
print(f"\n1. CORRELATION: Strong positive correlation (r = {pearson_r:.3f}) between")
print(f"   UV index and sunburn cases. UV explains ~{pearson_r**2*100:.1f}% of variance.")

print(f"\n2. SEASONALITY: Summer months show significantly higher sunburn rates")
print(f"   ({summer.mean():.1f} cases/week) compared to winter ({winter.mean():.1f} cases/week).")
print(f"\n3. PREDICTION: Each 1-unit increase in UV index is associated with")
print(f"   ~{model.params[1]:.1f} additional sunburn cases per week.")

print("\n4. DATA LIMITATION: UV values appear to be 24-hour averages (max < 2.0),")
print("   which include nighttime zeros and dilute peak exposure.")

print("\n" + "="*70)
print("CONCLUSION")
print("="*70)
print("\n The statistical evidence supports a relationship between UV index")
print(" and emergency department sunburn cases in England, though the UV")
print(" metric is a 24-hour average (values < 2.0), diluting")
print(" peak intensity typically required for physiological damage.")